# 1  Imports and Settings

In [1]:

import torch
from datasets import load_dataset, interleave_datasets
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    TrainingArguments, 
    Trainer, 
    DataCollatorForLanguageModeling
)

# --- Configuration ---
MODEL_PATH = "./gemma3-270M-kin-en-64k_model" # Path where you saved the resized model
OUTPUT_DIR = "./gemma3-141M-kinyarwanda-cpt"
SEQ_LENGTH = 2048       # Length of token chunks fed to the model
BATCH_SIZE = 4          # Adjust based on VRAM (8 with 2048 seq length should easily fit in 24GB)
GRAD_ACCUM = 4          # Effective batch size = 8 * 4 = 32
LEARNING_RATE = 2e-4    # Standard CPT learning rate
MAX_STEPS = 10000       # How long to train (adjust based on your dataset size)

In [2]:
import wandb
wandb.login()


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/xmikelinux/.netrc.
wandb: Currently logged in as: almugabo to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Step 2: Load the Model and Tokenizer

We load the modified model and tokenizer you prepared in the previous steps.


In [3]:


# Cell 2: Load Model and Tokenizer
print("Loading tokenizer and model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, 
    torch_dtype=torch.bfloat16, # Leverage RTX 4090's bf16 capabilities
    device_map="auto"           # Automatically place on your GPU
)

# Ensure pad token is set (crucial for batching)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

print(f"Model loaded. Total parameters: {model.num_parameters():,}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading tokenizer and model...


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Model loaded. Total parameters: 141,286,016


## Step 3: Prepare the Bilingual Data Stream

We need to load both datasets and interleave them. To train efficiently, we don't just feed the model sentence by sentence. Instead, we concatenate all the text and chop it up into uniform blocks of SEQ_LENGTH (e.g., 2048 tokens).



In [4]:

# Cell 3: Data Loading and Packing
print("Setting up datasets...")

# 1. Load datasets in streaming mode so we don't blow up RAM
ds_rw = load_dataset("mbazaNLP/kinyarwanda_monolingual_v01.1", split="train", streaming=True)
ds_en = load_dataset("HuggingFaceFW/fineweb", name="CC-MAIN-2025-26", split="train", streaming=True)


# 2. Mix them (e.g., 70% Kinyarwanda, 30% English)
mixed_ds = interleave_datasets([ds_rw, ds_en], probabilities=[0.7, 0.3], seed=42)

# 3. Tokenize and pack into fixed-length blocks
def tokenize_and_pack(examples):
    # Tokenize the texts
    tokenized = tokenizer(
        examples["text"], 
        truncation=False, 
        padding=False,
        return_attention_mask=False
    )
    
    # Concatenate all token IDs together
    all_ids = []
    for ids in tokenized["input_ids"]:
        all_ids.extend(ids)
        all_ids.append(tokenizer.eos_token_id) # Separate documents with EOS
        
    # Chop into chunks of SEQ_LENGTH
    total_length = len(all_ids)
    total_length = (total_length // SEQ_LENGTH) * SEQ_LENGTH # Drop the uneven tail
    
    result = {
        "input_ids": [all_ids[i : i + SEQ_LENGTH] for i in range(0, total_length, SEQ_LENGTH)]
    }
    return result

# Apply the packing mapping. We use batched=True to process multiple rows at once.
packed_ds = mixed_ds.map(
    tokenize_and_pack,
    batched=True,
    remove_columns=list(next(iter(mixed_ds)).keys()) # Remove raw text columns
)

Setting up datasets...


Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/250 [00:00<?, ?it/s]

## Create a Fixed Validation Dataset

In [5]:
print("Building fixed validation dataset...")

# Take the first 500 examples from the Kinyarwanda stream for validation
val_rw = load_dataset("mbazaNLP/kinyarwanda_monolingual_v01.0", split="train", streaming=True).take(500)
# Take 200 from English
val_en = load_dataset("HuggingFaceFW/fineweb", name="CC-MAIN-2025-26", split="train", streaming=True).take(200)

# Combine, tokenize, and pack them exactly like you did for the training set
# (Assuming you still have the `tokenize_and_pack` function from earlier)
val_mixed = interleave_datasets([val_rw, val_en])
packed_val_ds = val_mixed.map(
    tokenize_and_pack,
    batched=True,
    remove_columns=list(next(iter(val_mixed)).keys())
)

# Convert the iterable to a standard list-backed dataset so eval is fast and consistent
# We'll take exactly 100 packed sequences of length 2048
eval_dataset = list(packed_val_ds.take(100))

Building fixed validation dataset...


Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/250 [00:00<?, ?it/s]

# configure wandb 

In [6]:
import wandb

# Initialize the wandb run
wandb.init(
    project="gemma-3-kinyarwanda", # The overarching project name
    name="cpt-batch4-run-test-1",        # The specific name for this training run
    notes="Continuous pre-training with 70/30 mix on RTX 4090",
    tags=["CPT", "gemma3", "kinyarwanda"],
    config={
        "batch_size": 4,
        "seq_length": 2048,
        "vocab_size": len(tokenizer)
    }
)

## The Text Generation Callback

Instead of manually saving files, we can write a custom Hugging Face Callback. Every time the model runs an evaluation cycle, this callback will pause, generate text from your prompts, and log the text directly to a beautiful table in your WandB dashboard.

In [7]:
from transformers import TrainerCallback

class WandbGenerationCallback(TrainerCallback):
    def __init__(self, tokenizer, prompts):
        self.tokenizer = tokenizer
        self.prompts = prompts

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        # We only want to generate on the main process
        if not state.is_world_process_zero:
            return

        model.eval()
        print(f"\n--- Generating text at step {state.global_step} ---")
        
        # Create a WandB Table to store our generations
        table = wandb.Table(columns=["Step", "Prompt", "Generated Text"])
        
        for prompt in self.prompts:
            inputs = self.tokenizer(prompt, return_tensors="pt").to(model.device)
            
            # Generate up to 50 new tokens
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, 
                    max_new_tokens=50, 
                    do_sample=True,      # Add a little randomness 
                    temperature=0.7, 
                    pad_token_id=self.tokenizer.eos_token_id
                )
                
            # Decode the generated text
            generated_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # Add to the WandB table
            table.add_data(state.global_step, prompt, generated_text)
            print(f"Prompt: {prompt}\nOutput: {generated_text}\n")
            
        # Log the table to your WandB dashboard
        wandb.log({"evaluation/generations": table}, step=state.global_step)
        model.train() # Put model back in training mode

## Step 4: Setup Trainer and Start CPT

Finally, we use a standard data collator for causal language modeling (which automatically shifts inputs to create labels) and launch the Hugging Face Trainer.


In [8]:
test_prompts = [
    "Uyu munsi ni umunsi mwiza kuko",  # Fluency check
    "Amakuru yawe? Mfite",              # Conversational check
    "The capital of Rwanda is Kigali. In Kinyarwanda, this translates to" # Cross-lingual hint
]

In [9]:

# Cell 4: Training Setup and Execution
print("Initializing Trainer...")

# Data collator for next-token prediction (mlm=False means Causal LM)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps= 1500, #MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,                  # RTX 4090 loves bfloat16
    logging_steps=50,
    save_steps=2000,
    eval_strategy="steps",      # Evaluate every X steps
    eval_steps=50,             # Run validation every 500 steps
    report_to="wandb",           # Change to "none" if not 
    optim="adamw_torch"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=packed_ds,
    eval_dataset=eval_dataset,          # <-- Add the validation dataset    
    data_collator=data_collator,
    callbacks=[WandbGenerationCallback(tokenizer, test_prompts)] # <-- Add the callback    
)

print("Starting Continuous Pre-Training!")
trainer.train()
#trainer.train(resume_from_checkpoint=True)


##finish wandb 
# After trainer.train() finishes
wandb.finish()



# Save the final realigned model
trainer.save_model(f"{OUTPUT_DIR}/final_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_model")
print("Training complete and model saved.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Initializing Trainer...
Starting Continuous Pre-Training!


Step,Training Loss,Validation Loss
50,5.895201,4.687314
100,4.897949,4.329728
150,4.428519,4.099321
200,4.213155,3.992922
250,4.119456,3.919965
300,4.033669,3.827152
350,3.907838,3.805520
400,3.846824,3.755105
450,3.829965,3.706395
500,3.762309,3.669015



--- Generating text at step 50 ---
Prompt: Uyu munsi ni umunsi mwiza kuko
Output:  Uyu munsi ni umunsi mwiza kuko mu munsi bwo mu munsi w’s yari ari ari mu munsi by’’’u Rwanda. In our latest research, we discovered the existence of a new kind of DNA: The Double-Stranded Protein. This discovery opens up the possibility

Prompt: Amakuru yawe? Mfite
Output:  Amakuru yawe? Mfite. Mu rwego rw''''''''''''''''''''''''''''''''''''''''''''''

Prompt: The capital of Rwanda is Kigali. In Kinyarwanda, this translates to
Output:  The capital of Rwanda is Kigali. In Kinyarwanda, this translates to "The Bridge of Light," and it is also known as the "The Bridge of Light" or "The Bridge of Light."
The new approach is the result of the best research, including a co-investor, and the commitment of the


--- Generating text at step 100 ---
Prompt: Uyu munsi ni umunsi mwiza kuko
Output:  Uyu munsi ni umunsi mwiza kuko ari mu cyumweru gishize. Ni umuganza wa mbere kuko ari umuntu w’imyaka 15 w’imyaka 20, a

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


eval/loss,█▆▅▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/runtime,█▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/samples_per_second,▁▇▇▇▇▇████████████████████████
eval/steps_per_second,▁▇▇▇▇▇████████████████████████
train/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/global_step,▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/grad_norm,▅▂▂█▂▂▂▂▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▆█████▇▇▇▇▆▆▆▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁
train/loss,█▅▄▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,3.45343
eval/runtime,1.734


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete and model saved.
